In [ ]:
import re
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_groq import ChatGroq
from sentence_transformers import CrossEncoder

load_dotenv("../.env")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.load_local(
    "../data/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

all_docs = list(vector_store.docstore._dict.values())

bm25_retriever = BM25Retriever.from_documents(all_docs)
bm25_retriever.k = 10

print("Setup ready. Chunks:", vector_store.index.ntotal)


In [ ]:
def verify_answer(context, answer):
    check_prompt = f"""You are a strict checker.

Context:
{context}

Answer:
{answer}

Is every fact in the answer supported by the context above?
Reply with only one word: SUPPORTED or NOT_SUPPORTED"""
    result = llm.invoke(check_prompt)
    text = result.content.strip().upper()
    return "NOT_SUPPORTED" not in text and "SUPPORTED" in text


def make_queries(question):
    mq_prompt = f"""Write 3 different versions of this question using different words, so a search system can find the answer in help documents.
Question: {question}
Write only the 3 questions, one per line, no numbering."""
    text = llm.invoke(mq_prompt).content
    lines = []
    for l in text.strip().split("\n"):
        l = re.sub(r"^\d+[\.\)]\s*", "", l.strip()).strip("- ").strip()
        if l:
            lines.append(l)
    return [question] + lines[:3]


history = []

def rewrite_question(question):
    if not history:
        return question
    history_text = "\n".join(
        f"User: {q}\nAssistant: {a}" for q, a in history[-3:]
    )
    rw_prompt = f"""Given this conversation history:
{history_text}

Rewrite the follow-up question as a complete standalone question that can be understood without the history. Keep the same meaning. Write only the rewritten question.

Follow-up question: {question}"""
    return llm.invoke(rw_prompt).content.strip()


In [ ]:
THRESHOLD = 0.45

def ask(question):
    standalone = rewrite_question(question)
    if standalone != question:
        print("Samjha gaya sawal:", standalone)

    results = vector_store.similarity_search_with_relevance_scores(standalone, k=10)
    best_score = results[0][1]
    print("Best score:", round(best_score, 2))

    if best_score < THRESHOLD:
        print("Answer: I don't have this information in the documents.")
        print("Action: Escalate to human (low confidence)")
        history.append((question, "I don't have this information in the documents."))
        return

    queries = make_queries(standalone)

    pool = []
    seen_texts = set()
    for q in queries:
        docs_q = vector_store.similarity_search(q, k=5) + bm25_retriever.invoke(q)[:5]
        for d in docs_q:
            if d.page_content not in seen_texts:
                seen_texts.add(d.page_content)
                pool.append(d)
    print("Pool size:", len(pool))

    pairs = [(standalone, d.page_content) for d in pool]
    r_scores = reranker.predict(pairs)
    ranked = sorted(zip(pool, r_scores), key=lambda x: x[1], reverse=True)
    docs = [d for d, s in ranked[:3]]
    print("Top chunk after rerank:", docs[0].metadata["title"])

    context = "\n\n".join(d.page_content for d in docs)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: I don't have this information in the documents.

Context:
{context}

Question: {standalone}

Answer:"""

    answer = llm.invoke(prompt).content

    if answer.strip().startswith("I don't have this information"):
        print("\nAnswer:", answer)
        history.append((question, answer))
        return

    if verify_answer(context, answer):
        print("\nVerification: PASSED")
        print("\nAnswer:", answer)
    else:
        print("\nVerification: FAILED - Try one more time")
        strict_prompt = prompt + "\nBe very strict: use ONLY facts written in the context. Keep the answer short."
        answer2 = llm.invoke(strict_prompt).content

        if verify_answer(context, answer2):
            print("Verification (retry): PASSED")
            print("\nAnswer:", answer2)
            answer = answer2
        else:
            print("Verification (retry): FAILED")
            print("\nAnswer: I don't have this information in the documents.")
            print("Action: Escalate to human (answer could not be verified)")
            history.append((question, "I don't have this information in the documents."))
            return

    print("\nSources:")
    seen = set()
    for d in docs:
        if d.metadata["source"] not in seen:
            seen.add(d.metadata["source"])
            print("-", d.metadata["title"], "|", d.metadata["source"])

    history.append((question, answer))


In [ ]:
history.clear()
ask("How do I change my Notion plan?")
ask("And how do I get a refund?")
ask("What is Notion AI?")
ask("Is it free?")
